# Colab — snapshot smoke (500 видео, снапшоты каждые 15 мин)

Runbook: `Fetcher/docs/COLAB_20K_RUN_RU.md` (раздел snapshot smoke).

**Цель:** discover 500 видео → workers (download/enrich/HF) → снапшоты `1,2,3` через **15 мин** после `snapshot_0`.

**HF repos:** `Ilialebedev/dataset_snapshot_smoke_*` (отдельно от боевого 20k).

**HF_TOKEN** — Colab Secret `HF_TOKEN`.

Рекомендуется `output_dir` на `/content/...` (не Drive), чтобы mp4 не попадали в корзину Drive.

## 0. CONFIG

In [ ]:
# === отредактируй при необходимости ===
CONFIG = {
    "repo_url": "https://github.com/lebedev-ilia/TrendFlowML.git",
    "fetcher": "/content/TrendFlowML/Fetcher",
    "drive_secrets": "/content/drive/MyDrive/trendflow_secrets",
    "hf_repo_prefix": "Ilialebedev",
    "discover_limit": 500,
    "snapshot_interval_minutes": 15,
    "interval_sec": 120,
    "metrics_discover": 9095,
    "metrics_workers": 9096,
    "enable_tiktok": False,
    "enable_rutube": False,
    "enable_instagram": False,
}
CONFIG["output_dir"] = "/content/dataset_runs/snapshot-smoke-500"
CONFIG["worker_id"] = "colab-snapshot-main"
CONFIG["snapshot_sleep_seconds"] = CONFIG["snapshot_interval_minutes"] * 60

import os
from pathlib import Path

FETCHER = Path(CONFIG["fetcher"])
OUTPUT_DIR = Path(CONFIG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("output_dir:", OUTPUT_DIR)
print("discover_limit:", CONFIG["discover_limit"])
print("snapshot every:", CONFIG["snapshot_interval_minutes"], "min -> sleep", CONFIG["snapshot_sleep_seconds"], "s")

## 1. Drive + git

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import subprocess
from pathlib import Path

dest = Path("/content/TrendFlowML")
if (dest / ".git").exists():
    subprocess.run(["git", "-C", str(dest), "pull"], check=False)
    print("git pull")
else:
    subprocess.run(["git", "clone", CONFIG["repo_url"], str(dest)], check=True)
    print("git clone")

## 2. Зависимости

In [ ]:
%cd /content/TrendFlowML/Fetcher
!apt-get update -qq && apt-get install -y -qq nodejs
!python -m pip install -q -U pip
!python -m pip install -q -r requirements.txt
!python -m pip install -q -U huggingface_hub yt-dlp "pytubefix>=9.5.0" nodejs-wheel-binaries google-api-python-client

## 3. Cookies, keys, HF_TOKEN

Secret **не обязателен**, если уже выполнил `os.environ["HF_TOKEN"] = "hf_..."` в этой сессии. Ячейку с `userdata.get` можно не запускать.

In [ ]:
import os
from pathlib import Path
import shutil

FETCHER = Path(CONFIG["fetcher"])
secrets = Path(CONFIG["drive_secrets"])
cookies = FETCHER / "fetcher/dataset_collector/cookies"
keys = FETCHER / "fetcher/dataset_collector/keys"
cookies.mkdir(parents=True, exist_ok=True)
keys.mkdir(parents=True, exist_ok=True)

for p in (secrets / "cookies").glob("*.txt"):
    shutil.copy2(p, cookies / p.name)
shutil.copy2(secrets / "keys.txt", keys / "keys.txt")

# Вариант A: Colab → Secrets → добавить имя HF_TOKEN (рекомендуется)
# Вариант B: один раз в этой сессии (не сохраняй ноутбук с токеном):
# os.environ["HF_TOKEN"] = "hf_..."

if not os.environ.get("HF_TOKEN", "").startswith("hf_"):
    try:
        from google.colab import userdata
        t = (userdata.get("HF_TOKEN") or "").strip()
        if t:
            os.environ["HF_TOKEN"] = t
    except Exception as exc:
        print("Colab Secret HF_TOKEN not used:", type(exc).__name__)

if not os.environ.get("HF_TOKEN", "").startswith("hf_"):
    raise RuntimeError(
        "HF_TOKEN not set. Add Colab Secret HF_TOKEN, or run: os.environ['HF_TOKEN'] = 'hf_...'"
    )
print("HF_TOKEN OK:", os.environ["HF_TOKEN"][:10] + "...")

# Subprocess discover/bootstrap не всегда видит os.environ ядра — пишем файл:
_hf_path = Path(CONFIG["output_dir"]) / ".hf_token"
_hf_path.parent.mkdir(parents=True, exist_ok=True)
_hf_path.write_text(os.environ["HF_TOKEN"].strip() + "\n", encoding="utf-8")
print("saved for subprocess:", _hf_path)

kf = keys / "keys.txt"
os.environ["FETCHER_YOUTUBE_DATA_API_KEYS"] = kf.read_text().strip().replace("\n", ",")
print("API keys:", len(os.environ["FETCHER_YOUTUBE_DATA_API_KEYS"].split(",")))

## 4. Сброс runtime config (после git pull)

`hf_token_env` в JSON = **`"HF_TOKEN"`** (имя переменной). Сам токен `hf_...` — только Colab Secret, не в файл.

In [ ]:
import json
from pathlib import Path

runtime = Path(CONFIG["output_dir"]) / "runtime_dataset_campaign_20k.json"
if runtime.exists():
    cfg = json.loads(runtime.read_text(encoding="utf-8"))
    token_field = (cfg.get("hf_token_env") or "").strip()
    if token_field.startswith("hf_") and len(token_field) > 20:
        cfg["hf_token_env"] = "HF_TOKEN"
        runtime.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding="utf-8")
        print("fixed hf_token_env (was pasted token -> HF_TOKEN)")
    else:
        runtime.unlink()
        print("removed", runtime)
else:
    print("no old runtime json")

## 5. Discover 500 (nohup)

Campaign: `dataset_campaign_snapshot_smoke.json` — `snapshot_schedule_minutes: [0, 15, 30, 45]`.

In [ ]:
import subprocess, os
from pathlib import Path

if not os.environ.get("HF_TOKEN", "").startswith("hf_"):
    raise RuntimeError("Сначала выполни §3 (HF_TOKEN) — discover subprocess его не увидит")

_hf = Path(CONFIG["output_dir"]) / ".hf_token"
_hf.parent.mkdir(parents=True, exist_ok=True)
_hf.write_text(os.environ["HF_TOKEN"].strip() + "\n", encoding="utf-8")
print("HF_TOKEN for bootstrap:", _hf, "exists:", _hf.is_file())

FETCHER = Path(CONFIG["fetcher"])
cmd = [
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile", "snapshot-smoke",
    "--role", "discover",
    "--output-dir", CONFIG["output_dir"],
    "--hf-repo-prefix", CONFIG["hf_repo_prefix"],
    "--limit", str(CONFIG["discover_limit"]),
    "--parallel-colab-count", "1",
    "--metrics-port", str(CONFIG["metrics_discover"]),
]
if CONFIG.get("enable_tiktok"):
    cmd.append("--enable-tiktok")
if CONFIG.get("enable_rutube"):
    cmd.append("--enable-rutube")
if CONFIG.get("enable_instagram"):
    cmd.append("--enable-instagram")

log = Path("/content/snapshot_discover_log.txt")
with open(log, "w") as fh:
    p = subprocess.Popen(cmd, cwd=str(FETCHER), stdout=fh, stderr=subprocess.STDOUT, env=os.environ.copy())
print("discover pid", p.pid, "->", log)
print("tail:", "!tail -30", log)

## 6. Workers (download + enrich + HF)

In [ ]:
import subprocess, os
from pathlib import Path

FETCHER = Path(CONFIG["fetcher"])
cmd = [
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile", "snapshot-smoke",
    "--role", "workers",
    "--output-dir", CONFIG["output_dir"],
    "--hf-repo-prefix", CONFIG["hf_repo_prefix"],
    "--interval", str(CONFIG["interval_sec"]),
    "--metrics-port", str(CONFIG["metrics_workers"]),
    "--worker-id", CONFIG["worker_id"],
    "--worker-shard-index", "0",
    "--worker-shard-count", "1",
    "--parallel-colab-count", "1",
]

log = Path("/content/snapshot_workers_log.txt")
with open(log, "w") as fh:
    p = subprocess.Popen(cmd, cwd=str(FETCHER), stdout=fh, stderr=subprocess.STDOUT, env=os.environ.copy())
print("workers pid", p.pid)
print("logs:", Path(CONFIG["output_dir"]) / "logs/workers")

## 7. Status (перед снапшотами)

Дождись `accepted` ≈ 500 и хотя бы частичного download. Проверь `manifest.json` и `state/inventory/summary.json`.

In [ ]:
import subprocess, os
from pathlib import Path

cmd = [
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile", "snapshot-smoke",
    "--role", "status",
    "--output-dir", CONFIG["output_dir"],
]
raise SystemExit(subprocess.call(cmd, cwd=str(Path(CONFIG["fetcher"])), env=os.environ.copy()))

## 8. Snapshot loop (15 мин между индексами 1, 2, 3)

Блокирует runtime ~45 мин (3×15 мин). Можно прервать Ctrl+C между проходами.

Индексы: `1` → +15 мин от discover, `2` → +30 мин, `3` → +45 мин (только **YouTube** поддерживает snapshots в адаптерах).

In [ ]:
import subprocess, os
from pathlib import Path

FETCHER = Path(CONFIG["fetcher"])
cmd = [
    "python", "scripts/colab_20k_bootstrap.py",
    "--campaign-profile", "snapshot-smoke",
    "--role", "snapshot-loop",
    "--output-dir", CONFIG["output_dir"],
    "--snapshot-sleep-seconds", str(CONFIG["snapshot_sleep_seconds"]),
]
print("+", " ".join(cmd))
raise SystemExit(subprocess.call(cmd, cwd=str(FETCHER), env=os.environ.copy()))

## 9. Один снапшот вручную (если loop не нужен)

In [ ]:
# import time
# time.sleep(CONFIG["snapshot_sleep_seconds"])
%cd /content/TrendFlowML/Fetcher
# !python scripts/colab_20k_bootstrap.py --campaign-profile snapshot-smoke --role snapshot --snapshot-index 1 --output-dir "{CONFIG['output_dir']}"

## 10. Аудит + сверка с HF

In [ ]:
from pathlib import Path
import json

%cd /content/TrendFlowML/Fetcher
out = Path(CONFIG["output_dir"])
!python scripts/audit_dataset_run.py {out} --out {out}/audit_snapshot.json --check-hf
!python scripts/compare_hf_run_state.py {out} --out {out}/hf_compare.json

import json
manifest = json.loads((out / "manifest.json").read_text())
print("snapshots collected:", manifest.get("counters", {}).get("snapshots"))
print("snapshot_shards:", len(manifest.get("snapshot_shards", [])))